In [ ]:
!pip install -q datasets transformers evaluate sentence-transformers openai pandas numpy tqdm python-dotenv  google-generativeai requests tqdm -q sacrebleu rouge_score bert_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.4 MB/s eta 0:00:00


In [ ]:
import os
import csv
import json
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
import evaluate
from sentence_transformers import SentenceTransformer, util
from peft import PeftModel

# Other Packages
import pandas as pd
import numpy as np
import os
from huggingface_hub import login
import getpass
from google.colab import userdata
from tqdm.auto import tqdm
import os
import csv
import os, json, time, math
from tqdm import tqdm
from openai import OpenAI
import re

In [ ]:
# mount Google Drive first
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import pandas as pd
from datasets import load_dataset, Dataset, DatasetDict
from huggingface_hub import login

SOURCE_REPO = "Lakshan2003/GPT-4.1-customerservice-evaldata"
TARGET_REPO = "Lakshan2003/GPT-4.1-customerservice-LLM-as-a-judge-data"
BASE_PATH = "/content/drive/MyDrive/fyp-2025/Datasets/TestData/llm-judge-data"
PUSH_PRIVATE = False   # True = private dataset
SEED = 42
BATCH_SIZE = 500

In [ ]:
from datasets import load_dataset

ds = load_dataset("Lakshan2003/GPT-4.1-customerservice-evaldata")
df = ds['train'].to_pandas()
df.head()

README.md:   0%|          | 0.00/541 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/42.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/36669 [00:00<?, ? examples/s]

,conversation_id,instruction,history,history_summary,client_question,ground_truth,generated_answer
0,66bf0e79e2e441a29732279350a46762,You are a professional call-center customer se...,"Agent: Hi, this is Maureen with CrestBank. How...",Paulette Smith contacted Maureen at CrestBank ...,"Yes, that would be great. Thank you.","I'm glad to help, Paulette. I'll need to speak...","You're very welcome, Paulette. I’ll send the t..."
1,cc2d1db36afe4911beaddfb701decf46,You are a professional call-center customer se...,"Agent: Hi, thank you for calling TrustPoint Ba...","Deric, a long-term customer of TrustPoint Bank...",That doesn't make sense sense. What do I have ...,"I understand your frustration, Deric. To quali...","I understand your frustration, Deric. To quali..."
2,91d0c0ecf6554515b04233268117458c,You are a professional call-center customer se...,"Agent: Hi, thank you for calling Innovation Fi...","Patrick, the client, expressed interest in Inn...",That's really convenient. Can you walk me thro...,"Sure, Patrick! I can help you with that. First...","Absolutely, Patrick! To get started, you’ll fi..."
3,f563000caa5b4a3ebf89c7f692c8e0e7,You are a professional call-center customer se...,"Agent: Good, thank you for calling Peak Financ...",Trent contacted Peak Financial Solutions to in...,"Hmm, I'm not sure. Can you give me some more i...","Sure, Trent. I can give you more details about...","Of course, Trent. If you choose the standard v..."
4,b27ee4ad2b1046e5ac6e06063c3508c0,You are a professional call-center customer se...,Agent: Good you for calling SummitBank. My nam...,Mr. Lane contacted SummitBank to file a claim ...,"Wait, why do I need to be transferred? Can't y...","I understand your concern, Mr. Lane. The claim...",I understand the process can feel a bit confus...


### Conversation Stage Segmentation

In [ ]:
from openai import OpenAI
import os
import pandas as pd
from tqdm import tqdm

SAVE_PATH = "/content/drive/MyDrive/fyp-2025/Datasets/TestData/llm-judge-data/conversation_stage_FULL.csv"
BATCH_SIZE = 500

api_key = userdata.get("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = api_key
client = OpenAI(api_key=api_key)


def classify_stage(history_text, client_question, ground_truth):
    prompt = f"""
You are a conversation stage classifier.

Use ALL provided information to determine the stage.

Stages:
- BEGINNING: greetings, customer explains their problem, initial intake.
- MIDDLE: back-and-forth discussion, clarifications, troubleshooting.
- END: resolution confirmed, closing remarks, customer satisfied or conversation is ending.

Return only one word:
BEGINNING / MIDDLE / END

Conversation History:
\"\"\"{history_text}\"\"\"

Client Question:
\"\"\"{client_question}\"\"\"

Ground Truth (Correct Response / Final Answer):
\"\"\"{ground_truth}\"\"\"
"""

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return response.choices[0].message.content.strip().upper()

In [ ]:
def process_in_batches(df):
    total = len(df)
    num_batches = (total // BATCH_SIZE) + 1

    for batch_index in range(num_batches):

        start = batch_index * BATCH_SIZE
        end = min(start + BATCH_SIZE, total)
        batch_df = df.iloc[start:end].copy()

        if len(batch_df) == 0:
            continue

        print(f"\nProcessing Batch {batch_index} → rows {start} to {end-1}")

        batch_df["conversation_stage"] = [
            classify_stage(hist, cq, gt)
            for hist, cq, gt in tqdm(
                zip(batch_df["history"], batch_df["client_question"], batch_df["ground_truth"]),
                total=len(batch_df),
                desc=f"Batch {batch_index}"
            )
        ]

        # Append mode:
        write_header = not os.path.exists(SAVE_PATH)
        batch_df.to_csv(SAVE_PATH, mode='a', index=False, header=write_header)

        print(f"[APPENDED] Batch {batch_index} → {SAVE_PATH}")

    print("\n All batches processed and appended successfully.")


# RUN
process_in_batches(df)


Processing Batch 0 → rows 0 to 499


Batch 0: 100%|██████████| 500/500 [05:06<00:00,  1.63it/s]


[APPENDED] Batch 0 → /content/drive/MyDrive/fyp-2025/Datasets/TestData/llm-judge-data/conversation_stage_FULL.csv

Processing Batch 1 → rows 500 to 999


Batch 1: 100%|██████████| 500/500 [04:59<00:00,  1.67it/s]


[APPENDED] Batch 1 → /content/drive/MyDrive/fyp-2025/Datasets/TestData/llm-judge-data/conversation_stage_FULL.csv

Processing Batch 2 → rows 1000 to 1499


...

[APPENDED] Batch 71 → /content/drive/MyDrive/fyp-2025/Datasets/TestData/llm-judge-data/conversation_stage_FULL.csv

Processing Batch 72 → rows 36000 to 36499


Batch 72: 100%|██████████| 500/500 [05:07<00:00,  1.63it/s]


[APPENDED] Batch 72 → /content/drive/MyDrive/fyp-2025/Datasets/TestData/llm-judge-data/conversation_stage_FULL.csv

Processing Batch 73 → rows 36500 to 36668


Batch 73: 100%|██████████| 169/169 [02:19<00:00,  1.21it/s]

[APPENDED] Batch 73 → /content/drive/MyDrive/fyp-2025/Datasets/TestData/llm-judge-data/conversation_stage_FULL.csv

✅ All batches processed and appended successfully.


In [ ]:
from datasets import Dataset
from huggingface_hub import login

# CONFIG
TARGET_REPO = "Lakshan2003/Client-Agent-Conversations-with-stages"
PUSH_PRIVATE = False   # Set True if you want private


print("\nLoading final merged dataset...")
final_df = pd.read_csv(SAVE_PATH)   # The file you appended batches to
print("Rows:", len(final_df))

# Convert to Hugging Face Dataset
hf_dataset = Dataset.from_pandas(final_df)

print("Pushing to Hugging Face Hub...")
hf_dataset.push_to_hub(
    repo_id=TARGET_REPO,
    private=PUSH_PRIVATE
)

print(f"\n Upload complete.")
print(f" View dataset: https://huggingface.co/datasets/{TARGET_REPO}")


Loading final merged dataset...
Rows: 36669
Pushing to Hugging Face Hub...


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/37 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  39%|###9      | 16.7MB / 42.7MB            

README.md:   0%|          | 0.00/588 [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.



 Upload complete.
 View dataset: https://huggingface.co/datasets/Lakshan2003/Client-Agent-Conversations-with-stages


#### LLM as a judge dataset Creation

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/fyp-2025/Datasets/TestData/llm-judge-data/conversation_stage_FULL.csv")

In [ ]:
print(df.columns.tolist())

['conversation_id', 'instruction', 'history', 'history_summary', 'client_question', 'ground_truth', 'generated_answer', 'conversation_stage']


In [ ]:
df.head()

,conversation_id,instruction,history,history_summary,client_question,ground_truth,generated_answer,conversation_stage
0,66bf0e79e2e441a29732279350a46762,You are a professional call-center customer se...,"Agent: Hi, this is Maureen with CrestBank. How...",Paulette Smith contacted Maureen at CrestBank ...,"Yes, that would be great. Thank you.","I'm glad to help, Paulette. I'll need to speak...","You're very welcome, Paulette. I’ll send the t...",MIDDLE
1,cc2d1db36afe4911beaddfb701decf46,You are a professional call-center customer se...,"Agent: Hi, thank you for calling TrustPoint Ba...","Deric, a long-term customer of TrustPoint Bank...",That doesn't make sense sense. What do I have ...,"I understand your frustration, Deric. To quali...","I understand your frustration, Deric. To quali...",MIDDLE
2,91d0c0ecf6554515b04233268117458c,You are a professional call-center customer se...,"Agent: Hi, thank you for calling Innovation Fi...","Patrick, the client, expressed interest in Inn...",That's really convenient. Can you walk me thro...,"Sure, Patrick! I can help you with that. First...","Absolutely, Patrick! To get started, you’ll fi...",MIDDLE
3,f563000caa5b4a3ebf89c7f692c8e0e7,You are a professional call-center customer se...,"Agent: Good, thank you for calling Peak Financ...",Trent contacted Peak Financial Solutions to in...,"Hmm, I'm not sure. Can you give me some more i...","Sure, Trent. I can give you more details about...","Of course, Trent. If you choose the standard v...",MIDDLE
4,b27ee4ad2b1046e5ac6e06063c3508c0,You are a professional call-center customer se...,Agent: Good you for calling SummitBank. My nam...,Mr. Lane contacted SummitBank to file a claim ...,"Wait, why do I need to be transferred? Can't y...","I understand your concern, Mr. Lane. The claim...",I understand the process can feel a bit confus...,MIDDLE


In [ ]:
df.shape

(36669, 8)

In [ ]:
import pandas as pd

TARGET_TOTAL = 6000
ratio_begin = 0.1
ratio_end = 0.1
ratio_middle = 1 - ratio_begin - ratio_end

n_begin = int(TARGET_TOTAL * ratio_begin)     # 600
n_end = int(TARGET_TOTAL * ratio_end)         # 600
n_middle = TARGET_TOTAL - n_begin - n_end     # 4800

print(n_begin, n_middle, n_end)

# Ensure class labels are uppercase consistently
df['conversation_stage'] = df['conversation_stage'].str.upper()

# Subsets
begin_df = df[df['conversation_stage'] == "BEGINNING"]
middle_df = df[df['conversation_stage'] == "MIDDLE"]
end_df = df[df['conversation_stage'] == "END"]

# Check class counts
print("Available counts:")
print(begin_df.shape[0], middle_df.shape[0], end_df.shape[0])

# Sample
sample_begin = begin_df.sample(n=n_begin, random_state=42)
sample_middle = middle_df.sample(n=n_middle, random_state=42)
sample_end = end_df.sample(n=n_end, random_state=42)

# Combine
balanced_df = pd.concat([sample_begin, sample_middle, sample_end]).sample(frac=1, random_state=42).reset_index(drop=True)

print("Final balanced size:", balanced_df.shape)

600 4800 600
Available counts:
4932 26125 5612
Final balanced size: (6000, 8)


In [ ]:
balanced_df["conversation_stage"].value_counts()

,count
conversation_stage,
MIDDLE,4800
BEGINNING,600
END,600


In [ ]:
print(balanced_df.head())

                    conversation_id  \
0  9fa135b5ea9d4bb691661bdcbc67d482   
1  81f14e9574cc42dba2f1c3745b5064a8   
2  9c1c012f15bc4cb7a8b7219967b3d64f   
3  1833b5ef87ba41f594105be104b9ae31   
4  56f7761934e246188695c4e440821a72   

                                         instruction  \
0  You are a professional call-center customer se...   
1  You are a professional call-center customer se...   
2  You are a professional call-center customer se...   
3  You are a professional call-center customer se...   
4  You are a professional call-center customer se...   

                                             history  \
0  Agent: Good morning, thank you for holding. My...   
1  Client: Hi Elsie, I'm calling because there's ...   
2  Agent: Hi there thank you for calling Innovati...   
3  Agent: Thank you for calling VitalBank. My nam...   
4  Agent: Hello, thank name is Terry. I'm calling...   

                                     history_summary  \
0  Ms. Doris contacted Junior at St

In [ ]:
# Make sure we work on the correct dataframe
df = balanced_df.copy()

# Current column order
cols = df.columns.tolist()

# Remove conversation_stage from its current position
cols.remove("conversation_stage")

# Insert it right after conversation_id
cols.insert(cols.index("conversation_id") + 1, "conversation_stage")

# Apply new order
df = df[cols]

print(df.columns.tolist())

['conversation_id', 'conversation_stage', 'instruction', 'history', 'history_summary', 'client_question', 'ground_truth', 'generated_answer']


In [ ]:
stage_map = balanced_df[["conversation_id", "conversation_stage"]].copy()
stage_map["conversation_stage"] = stage_map["conversation_stage"].str.upper().str.strip()

print(stage_map.head())
print("Stage map rows:", len(stage_map))

                    conversation_id conversation_stage
0  9fa135b5ea9d4bb691661bdcbc67d482             MIDDLE
1  81f14e9574cc42dba2f1c3745b5064a8             MIDDLE
2  9c1c012f15bc4cb7a8b7219967b3d64f          BEGINNING
3  1833b5ef87ba41f594105be104b9ae31             MIDDLE
4  56f7761934e246188695c4e440821a72             MIDDLE
Stage map rows: 6000


### Upload LLM-as-a-judge datasets

In [ ]:
stage_map

,conversation_id,conversation_stage
0,9fa135b5ea9d4bb691661bdcbc67d482,MIDDLE
1,81f14e9574cc42dba2f1c3745b5064a8,MIDDLE
2,9c1c012f15bc4cb7a8b7219967b3d64f,BEGINNING
3,1833b5ef87ba41f594105be104b9ae31,MIDDLE
4,56f7761934e246188695c4e440821a72,MIDDLE
...,...,...
5995,6da9fc52405c458db82675dd2157123d,MIDDLE
5996,3d42e69b4c6247c68ebe39e71e888824,MIDDLE
5997,d37cb85fa10d4600b5faff19ec79eaa8,MIDDLE
5998,0dd3797182c24046b9e1d4582450e155,MIDDLE


In [ ]:
from datasets import load_dataset, Dataset
import pandas as pd
import os

def process_model_dataset(source_repo, target_repo, save_dir):
    print(f"\nProcessing {source_repo}")

    # Derive model name from repo path (e.g., "Lakshan/model-name" → "model-name")
    model_name = source_repo.split("/")[-1]

    ds = load_dataset(source_repo)
    df = ds["train"].to_pandas()

    # Merge stage info (only keep rows that match conversation_ids in the map)
    merged = df.merge(stage_map, on="conversation_id", how="inner")

    print(f"Original size = {len(df)}, After filtering = {len(merged)}")

    #  REORDER COLUMNS
    cols = merged.columns.tolist()

    cols.remove("conversation_stage")
    cols.insert(cols.index("conversation_id") + 1, "conversation_stage")

    merged = merged[cols]

    # Add evaluation columns for LLM-as-a-judge scoring
    merged["Human-Likeness"] = ""
    merged["Continuity and Context Understanding"] = ""
    merged["Tone and Clarity"] = ""
    merged["Task Appropriateness"] = ""

    #  SAVE TO GOOGLE DRIVE
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f"{model_name}.csv")
    merged.to_csv(save_path, index=False)
    print(f"CSV saved locally/drive → {save_path}")

    #  UPLOAD TO HUGGING FACE HUB
    hf_ds = Dataset.from_pandas(merged)
    hf_ds.push_to_hub(repo_id=target_repo, private=False)

    print(f"Uploaded → https://huggingface.co/datasets/{target_repo}")

In [ ]:
model_repos = {
    "Gemma3-4B-instruct":      "Lakshan2003/Gemma3-4B-instruct-customerservice-evaldata",
    "gemini-2.5-flash":        "Lakshan2003/gemini-2.5-flash-customerservice-evaldata",
    "Phi-4-mini":              "Lakshan2003/Phi-4-mini-customerservice-evaldata",
    "Llama3.2-3B-instruct":       "Lakshan2003/Llama3.2-3B-instruct-customerservice-evaldata",
    "SmolLM3-3B":              "Lakshan2003/SmolLM3-3B-customerservice-evaldata",
    "GPT-4.1":                 "Lakshan2003/GPT-4.1-customerservice-evaldata",
    "Virtuoso-large":          "Lakshan2003/Virtuoso-large-customerservice-evaldata",
    "Qwen3-4B":                "Lakshan2003/Qwen3-4B-customerservice-evaldata",
    "Qwen3-8B":              "Lakshan2003/Qwen3-8B-customerservice-evaldata",
    "Llama3.1-8b": "Lakshan2003/Llama3.1-8b-instruct-customerservice-evaldata",
    "Llama3.2-1b": "Lakshan2003/Llama3.2-1B-instruct-customerservice-evaldata",
    "Qwen3-1.7B": "Lakshan2003/Qwen3-1.7B-customerservice-evaldata",
}

save_dir = "/content/drive/MyDrive/fyp-2025/Datasets/LLMJudgeData"

for model_name, source_repo in model_repos.items():
    target_repo = f"Lakshan2003/{model_name}-customerservice-LLM-as-a-judge-data"
    process_model_dataset(source_repo, target_repo, save_dir)


Processing Lakshan2003/Gemma3-4B-instruct-customerservice-evaldata


README.md:   0%|          | 0.00/541 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/43.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/36669 [00:00<?, ? examples/s]

Original size = 36669, After filtering = 6000
CSV saved locally/drive → /content/drive/MyDrive/fyp-2025/Datasets/LLMJudgeData/Gemma3-4B-instruct-customerservice-evaldata.csv


...

Creating parquet from Arrow format:   0%|          | 0/6 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  97%|#########6| 6.62MB / 6.85MB            

Uploaded → https://huggingface.co/datasets/Lakshan2003/Qwen3-1.7B-customerservice-LLM-as-a-judge-data


#### Human As a judge Dataset

In [ ]:
import pandas as pd

# Load data
df = pd.read_csv("/content/drive/MyDrive/fyp-2025/Datasets/TestData/llm-judge-data/conversation_stage_FULL.csv")

# Ensure class labels are consistent
df['conversation_stage'] = df['conversation_stage'].str.upper().str.strip()

# Fixed ratios
TARGET_TOTAL = 500
ratio_begin = 0.10
ratio_end = 0.10
ratio_middle = 1 - ratio_begin - ratio_end  # 0.80

n_begin = int(TARGET_TOTAL * ratio_begin)   # 50
n_end = int(TARGET_TOTAL * ratio_end)       # 50
n_middle = TARGET_TOTAL - n_begin - n_end   # 400

print("Sample counts →", n_begin, n_middle, n_end)

# Split dataset by stage
begin_df = df[df['conversation_stage'] == "BEGINNING"]
middle_df = df[df['conversation_stage'] == "MIDDLE"]
end_df = df[df['conversation_stage'] == "END"]

print("Available counts →", begin_df.shape[0], middle_df.shape[0], end_df.shape[0])

Sample counts → 50 400 50
Available counts → 4932 26125 5612


In [ ]:
# Sample with reproducibility
sample_begin = begin_df.sample(n=n_begin, random_state=42, replace=False)
sample_middle = middle_df.sample(n=n_middle, random_state=42, replace=False)
sample_end = end_df.sample(n=n_end, random_state=42, replace=False)

# Combine + Shuffle
human_eval_df = pd.concat([sample_begin, sample_middle, sample_end]) \
                   .sample(frac=1, random_state=42) \
                   .reset_index(drop=True)

print("Final dataset size:", human_eval_df.shape)
print(human_eval_df["conversation_stage"].value_counts())

Final dataset size: (500, 8)
conversation_stage
MIDDLE       400
END           50
BEGINNING     50
Name: count, dtype: int64


In [ ]:
human_eval_df["conversation_stage"].value_counts()

,count
conversation_stage,
MIDDLE,400
END,50
BEGINNING,50


In [ ]:
print(human_eval_df.head())

                    conversation_id  \
0  ac16e172ac7c430693afdd68952aff69   
1  018ccc55eb5049f9be7a9c4310be4ce4   
2  16b785468c884d849b34af277a2b0ab3   
3  4b4c17a7d68d4c8da33591e7872e3a9d   
4  90f7d6976ac34aa285cf69fdded293fe   

                                         instruction  \
0  You are a professional call-center customer se...   
1  You are a professional call-center customer se...   
2  You are a professional call-center customer se...   
3  You are a professional call-center customer se...   
4  You are a professional call-center customer se...   

                                             history  \
0  Agent: Hi, thank you for calling Community Fin...   
1  Client: Hi, I needm calling about my student a...   
2  Agent: Good morning, thank you for calling Tru...   
3  Agent: Hi, thank you for calling Future Financ...   
4  Agent: Hello, thank you for calling Cornerston...   

                                     history_summary  \
0  Jackeline contacted Community Fi

In [ ]:
# Make sure we work on the correct dataframe
df = human_eval_df.copy()

# Current column order
cols = df.columns.tolist()

# Remove conversation_stage from its current position
cols.remove("conversation_stage")

# Insert it right after conversation_id
cols.insert(cols.index("conversation_id") + 1, "conversation_stage")

# Apply new order
df = df[cols]

print(df.columns.tolist())

['conversation_id', 'conversation_stage', 'instruction', 'history', 'history_summary', 'client_question', 'ground_truth', 'generated_answer']


In [ ]:
stage_map = human_eval_df[["conversation_id", "conversation_stage"]].copy()
stage_map["conversation_stage"] = stage_map["conversation_stage"].str.upper().str.strip()

print(stage_map.head())
print("Stage map rows:", len(stage_map))

                    conversation_id conversation_stage
0  ac16e172ac7c430693afdd68952aff69             MIDDLE
1  018ccc55eb5049f9be7a9c4310be4ce4             MIDDLE
2  16b785468c884d849b34af277a2b0ab3             MIDDLE
3  4b4c17a7d68d4c8da33591e7872e3a9d             MIDDLE
4  90f7d6976ac34aa285cf69fdded293fe             MIDDLE
Stage map rows: 500


#### Upload Human Eval Datasets

In [ ]:
from datasets import load_dataset, Dataset
import pandas as pd
import os

def process_model_dataset(model_name, source_repo, target_repo, save_dir):
    print(f"\nProcessing {model_name} ({source_repo})")

    ds = load_dataset(source_repo)
    df = ds["train"].to_pandas()

    # Merge stage info
    merged = df.merge(stage_map, on="conversation_id", how="inner")
    print(f"Original size = {len(df)}, After filtering = {len(merged)}")

    # Reorder columns
    cols = merged.columns.tolist()
    cols.remove("conversation_stage")
    cols.insert(cols.index("conversation_id") + 1, "conversation_stage")
    merged = merged[cols]

    # Add evaluation columns (initialize as empty)
    merged["Human-Likeness"] = ""
    merged["Continuity and Context Understanding"] = ""
    merged["Tone and Clarity"] = ""
    merged["Task Appropriateness"] = ""

    #  SAVE TO GOOGLE DRIVE using model_name
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f"{model_name}.csv")
    merged.to_csv(save_path, index=False)
    print(f"CSV saved to: {save_path}")

    # Push to Hugging Face Hub
    hf_ds = Dataset.from_pandas(merged)
    hf_ds.push_to_hub(repo_id=target_repo, private=False)

    print(f"Uploaded → https://huggingface.co/datasets/{target_repo}")

In [ ]:
from datasets import load_dataset, Dataset
import pandas as pd
import os
from openpyxl import load_workbook
from openpyxl.styles import Alignment

def process_model_dataset(model_name, source_repo, target_repo, save_dir):
    print(f"\nProcessing {model_name} ({source_repo})")

    ds = load_dataset(source_repo)
    df = ds["train"].to_pandas()

    # Merge stage info
    merged = df.merge(stage_map, on="conversation_id", how="inner")
    print(f"Original size = {len(df)}, After filtering = {len(merged)}")

    # Reorder columns
    cols = merged.columns.tolist()
    cols.remove("conversation_stage")
    cols.insert(cols.index("conversation_id") + 1, "conversation_stage")
    merged = merged[cols]

    # Add evaluation columns
    merged["Human-Likeness"] = ""
    merged["Continuity and Context Understanding"] = ""
    merged["Tone and Clarity"] = ""
    merged["Task Appropriateness"] = ""

    os.makedirs(save_dir, exist_ok=True)

    # Save CSV
    csv_path = os.path.join(save_dir, f"{model_name}.csv")
    merged.to_csv(csv_path, index=False)
    print(f"CSV saved to: {csv_path}")

    # Save Excel
    xlsx_path = os.path.join(save_dir, f"{model_name}_readable.xlsx")
    merged.to_excel(xlsx_path, index=False)

    wb = load_workbook(xlsx_path)
    ws = wb.active

    # Auto-fit width + wrap text
    for col in ws.columns:
        max_len = 0
        col_letter = col[0].column_letter
        for cell in col:
            cell.alignment = Alignment(wrap_text=True)
            if cell.value:
                max_len = max(max_len, len(str(cell.value)))
        ws.column_dimensions[col_letter].width = min(max_len + 2, 80)

    wb.save(xlsx_path)
    print(f"Readable Excel saved to: {xlsx_path}")

    # Upload to HF
    hf_ds = Dataset.from_pandas(merged)
    hf_ds.push_to_hub(repo_id=target_repo, private=False)

    print(f"Uploaded → https://huggingface.co/datasets/{target_repo}")

In [ ]:
model_repos = {
    "Gemma3-4B-instruct":      "Lakshan2003/Gemma3-4B-instruct-customerservice-evaldata",
    "gemini-2.5-flash":        "Lakshan2003/gemini-2.5-flash-customerservice-evaldata",
    "Phi-4-mini":              "Lakshan2003/Phi-4-mini-customerservice-evaldata",
    "Llama3.2-instruct":       "Lakshan2003/Llama3.2-instruct-customerservice-evaldata",
    "SmolLM3-3B":              "Lakshan2003/SmolLM3-3B-customerservice-evaldata",
    "GPT-4.1":                 "Lakshan2003/GPT-4.1-customerservice-evaldata",
    "Virtuoso-large":          "Lakshan2003/Virtuoso-large-customerservice-evaldata",
    "Qwen3-4B":                "Lakshan2003/Qwen3-4B-customerservice-evaldata"
}


save_dir = "/content/drive/MyDrive/fyp-2025/Datasets/HumanEvalData"

for model_name, source_repo in model_repos.items():
    target_repo = f"Lakshan2003/{model_name}-customerservice-Human-eval-data"
    process_model_dataset(model_name, source_repo, target_repo, save_dir)



Processing Gemma3-4B-instruct (Lakshan2003/Gemma3-4B-instruct-customerservice-evaldata)


README.md:   0%|          | 0.00/541 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/43.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/36669 [00:00<?, ? examples/s]

Original size = 36669, After filtering = 500
CSV saved to: /content/drive/MyDrive/fyp-2025/Datasets/HumanEvalData/Gemma3-4B-instruct.csv
Readable Excel saved to: /content/drive/MyDrive/fyp-2025/Datasets/HumanEvalData/Gemma3-4B-instruct_readable.xlsx


...

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########|  572kB /  572kB            

README.md:   0%|          | 0.00/782 [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


Uploaded → https://huggingface.co/datasets/Lakshan2003/Qwen3-4B-customerservice-Human-eval-data


#### LLM as a judge task results

In [ ]:
from datasets import load_dataset, Dataset
import pandas as pd


# LLM-as-a-Judge datasets
DATASETS = {
    "Gemma3-4B": "Lakshan2003/Gemma3-4B-instruct-customerservice-LLM-as-a-judge-data",
    "Gemini-2.5-Flash": "Lakshan2003/gemini-2.5-flash-customerservice-LLM-as-a-judge-data",
    "Phi-4-Mini": "Lakshan2003/Phi-4-mini-customerservice-LLM-as-a-judge-data",
    "Llama-3.2-3B": "Lakshan2003/Llama3.2-3B-instruct-customerservice-LLM-as-a-judge-data",
    "SmolLM3-3B": "Lakshan2003/SmolLM3-3B-customerservice-LLM-as-a-judge-data",
    "GPT-4.1": "Lakshan2003/GPT-4.1-customerservice-LLM-as-a-judge-data",
    "Virtuoso-Large": "Lakshan2003/Virtuoso-large-customerservice-LLM-as-a-judge-data",
    "Qwen3-4B": "Lakshan2003/Qwen3-4B-customerservice-LLM-as-a-judge-data",
    "Qwen3-8B":              "Lakshan2003/Qwen3-8B-customerservice-LLM-as-a-judge-data",
    "Llama3.1-8b": "Lakshan2003/Llama3.1-8b-instruct-customerservice-LLM-as-a-judge-data",
    "Llama3.2-1b": "Lakshan2003/Llama3.2-1B-instruct-customerservice-LLM-as-a-judge-data",
    "Qwen3-1.7B": "Lakshan2003/Qwen3-1.7B-customerservice-LLM-as-a-judge-data",
}


TASK_COLUMNS = [
    "Human-Likeness",
    "Continuity and Context Understanding",
    "Tone and Clarity",
    "Task Appropriateness",
]

rows = []


# Aggregate task-wise means
for model_name, repo_id in DATASETS.items():
    ds = load_dataset(repo_id, split="train")
    df = ds.to_pandas()

    # Ensure numeric values
    df[TASK_COLUMNS] = df[TASK_COLUMNS].apply(
        pd.to_numeric, errors="coerce"
    )

    task_means = df[TASK_COLUMNS].mean().round(4)
    overall_mean = round(task_means.mean(), 4)
    count = len(df)

    row = {
        "model": model_name,
        **task_means.to_dict(),
        "Overall_Mean": overall_mean,
        "Tested Conversation Count": count,
        "Judge": "claude-sonnet-4-5",
    }

    rows.append(row)

# Final dataframe (8 columns)
final_df = pd.DataFrame(rows)
print(final_df)

# Sort by Overall_Mean (descending = best first)
final_df = final_df.sort_values(
    by="Overall_Mean",
    ascending=False
).reset_index(drop=True)

# Push aggregated dataset to Hugging Face Hub
final_dataset = Dataset.from_pandas(final_df)

final_dataset.push_to_hub(
    "Lakshan2003/customerservice-llm-as-a-judge-task-results",
    private=False
)

README.md:   0%|          | 0.00/790 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/7.15M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6000 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/790 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/6.85M [00:00<?, ?B/s]

...

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 4.61kB / 4.61kB            

README.md:   0%|          | 0.00/599 [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/datasets/Lakshan2003/customerservice-llm-as-a-judge-task-results/commit/f5188263cf7471342273020b3a1dae1aded572ff', commit_message='Upload dataset', commit_description='', oid='f5188263cf7471342273020b3a1dae1aded572ff', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Lakshan2003/customerservice-llm-as-a-judge-task-results', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Lakshan2003/customerservice-llm-as-a-judge-task-results'), pr_revision=None, pr_num=None)

In [ ]:
final_df

,model,Human-Likeness,Continuity and Context Understanding,Tone and Clarity,Task Appropriateness,Overall_Mean,Tested Conversation Count,Judge
0,GPT-4.1,4.3158,4.0788,4.3808,3.8078,4.1458,6000,claude-sonnet-4-5
1,Virtuoso-Large,4.1705,3.8640,4.2038,3.5302,3.9421,6000,claude-sonnet-4-5
2,Llama3.1-8b,4.1149,3.5914,4.1488,3.3224,3.7944,6000,claude-sonnet-4-5
3,Gemini-2.5-Flash,4.0542,3.7418,4.1013,3.1802,3.7694,6000,claude-sonnet-4-5
4,Qwen3-8B,3.9502,3.6482,4.0668,3.3055,3.7427,6000,claude-sonnet-4-5
5,Llama-3.2-3B,4.0752,3.4797,4.1052,3.2125,3.7182,6000,claude-sonnet-4-5
6,Qwen3-4B,4.0440,3.4303,4.0712,3.1695,3.6788,6000,claude-sonnet-4-5
7,Phi-4-Mini,3.9877,3.3592,4.0340,3.0930,3.6185,6000,claude-sonnet-4-5
8,Qwen3-1.7B,3.7380,3.3622,3.8176,2.9935,3.4778,6000,claude-sonnet-4-5
9,Llama3.2-1b,3.1648,2.3575,3.3422,2.1713,2.7590,6000,claude-sonnet-4-5


#### Stage wise LLM as a judge Results

In [ ]:
from datasets import load_dataset
import pandas as pd

STAGE_COL = "conversation_stage"

rows = []

for model_name, repo in DATASETS.items():
    ds = load_dataset(repo, split="train")
    df = ds.to_pandas()

    df["Model"] = model_name
    rows.append(df)

llm_judge_df = pd.concat(rows, ignore_index=True)

Repo card metadata block was not found. Setting CardData to empty.


In [ ]:
def llm_judge_stage_results_df(df):
    grouped = (
        df
        .groupby(["Model", STAGE_COL])[TASK_COLUMNS]
        .mean()
        .round(3)
        .reset_index()
    )

    grouped["Overall_Mean"] = (
        grouped[TASK_COLUMNS].mean(axis=1).round(3)
    )

    # count UNIQUE conversations per stage
    counts = (
        df
        .groupby(["Model", STAGE_COL])["conversation_id"]
        .nunique()
        .reset_index(name="Tested Conversation Count")
    )

    result = grouped.merge(
        counts,
        on=["Model", STAGE_COL]
    )

    return result

In [ ]:
llm_judge_stagewise_df = llm_judge_stage_results_df(llm_judge_df)

In [ ]:
llm_judge_stagewise_df

,Model,conversation_stage,Human-Likeness,Continuity and Context Understanding,Tone and Clarity,Task Appropriateness,Overall_Mean,Tested Conversation Count
0,GPT-4.1,BEGINNING,4.310,4.018,4.383,3.715,4.106,600
1,GPT-4.1,END,4.563,4.323,4.588,4.253,4.432,600
2,GPT-4.1,MIDDLE,4.286,4.056,4.355,3.764,4.115,4800
3,Gemini-2.5-Flash,BEGINNING,4.143,3.768,4.185,3.350,3.862,600
4,Gemini-2.5-Flash,END,4.085,3.577,4.177,3.433,3.818,600
5,Gemini-2.5-Flash,MIDDLE,4.039,3.759,4.081,3.127,3.752,4800
6,Gemma3-4B,BEGINNING,2.517,1.553,2.557,1.508,2.034,600
7,Gemma3-4B,END,2.363,1.843,2.418,1.945,2.142,600
8,Gemma3-4B,MIDDLE,2.618,1.737,2.624,1.659,2.160,4800
9,Llama-3.2-3B,BEGINNING,4.125,3.625,4.160,3.288,3.800,600


### Stage based evaluation datasets

In [ ]:
llm_judge_stagewise_df.sort_values(
    ["conversation_stage", "Overall_Mean"]
)

,Model,conversation_stage,Human-Likeness,Continuity and Context Understanding,Tone and Clarity,Task Appropriateness,Overall_Mean,Tested Conversation Count
30,SmolLM3-3B,BEGINNING,2.227,1.578,2.290,1.503,1.900,600
6,Gemma3-4B,BEGINNING,2.517,1.553,2.557,1.508,2.034,600
15,Llama3.2-1b,BEGINNING,3.625,2.927,3.725,2.583,3.215,600
21,Qwen3-1.7B,BEGINNING,3.880,3.508,3.957,3.145,3.622,600
27,Qwen3-8B,BEGINNING,3.932,3.513,4.038,3.175,3.665,600
18,Phi-4-Mini,BEGINNING,4.058,3.473,4.113,3.158,3.700,600
24,Qwen3-4B,BEGINNING,4.100,3.575,4.130,3.253,3.764,600
9,Llama-3.2-3B,BEGINNING,4.125,3.625,4.160,3.288,3.800,600
12,Llama3.1-8b,BEGINNING,4.152,3.613,4.202,3.310,3.819,600
3,Gemini-2.5-Flash,BEGINNING,4.143,3.768,4.185,3.350,3.862,600


In [ ]:
from datasets import Dataset

# Ensure correct ordering inside each stage
begin_df = (
    llm_judge_stagewise_df[llm_judge_stagewise_df["conversation_stage"] == "BEGINNING"]
    .sort_values("Overall_Mean", ascending=False)
    .reset_index(drop=True)
)

middle_df = (
    llm_judge_stagewise_df[llm_judge_stagewise_df["conversation_stage"] == "MIDDLE"]
    .sort_values("Overall_Mean", ascending=False)
    .reset_index(drop=True)
)

end_df = (
    llm_judge_stagewise_df[llm_judge_stagewise_df["conversation_stage"] == "END"]
    .sort_values("Overall_Mean", ascending=False)
    .reset_index(drop=True)
)

In [ ]:
begin_df

,Model,conversation_stage,Human-Likeness,Continuity and Context Understanding,Tone and Clarity,Task Appropriateness,Overall_Mean,Tested Conversation Count
0,GPT-4.1,BEGINNING,4.310,4.018,4.383,3.715,4.106,600
1,Virtuoso-Large,BEGINNING,4.157,3.750,4.185,3.425,3.879,600
2,Gemini-2.5-Flash,BEGINNING,4.143,3.768,4.185,3.350,3.862,600
3,Llama3.1-8b,BEGINNING,4.152,3.613,4.202,3.310,3.819,600
4,Llama-3.2-3B,BEGINNING,4.125,3.625,4.160,3.288,3.800,600
5,Qwen3-4B,BEGINNING,4.100,3.575,4.130,3.253,3.764,600
6,Phi-4-Mini,BEGINNING,4.058,3.473,4.113,3.158,3.700,600
7,Qwen3-8B,BEGINNING,3.932,3.513,4.038,3.175,3.665,600
8,Qwen3-1.7B,BEGINNING,3.880,3.508,3.957,3.145,3.622,600
9,Llama3.2-1b,BEGINNING,3.625,2.927,3.725,2.583,3.215,600


In [ ]:
middle_df

,Model,conversation_stage,Human-Likeness,Continuity and Context Understanding,Tone and Clarity,Task Appropriateness,Overall_Mean,Tested Conversation Count
0,GPT-4.1,MIDDLE,4.286,4.056,4.355,3.764,4.115,4800
1,Virtuoso-Large,MIDDLE,4.137,3.835,4.173,3.470,3.904,4800
2,Gemini-2.5-Flash,MIDDLE,4.039,3.759,4.081,3.127,3.752,4800
3,Llama3.1-8b,MIDDLE,4.077,3.525,4.111,3.236,3.737,4800
4,Qwen3-8B,MIDDLE,3.924,3.625,4.052,3.249,3.712,4800
5,Llama-3.2-3B,MIDDLE,4.034,3.388,4.065,3.104,3.648,4800
6,Qwen3-4B,MIDDLE,4.001,3.342,4.030,3.061,3.609,4800
7,Phi-4-Mini,MIDDLE,3.943,3.267,3.991,2.980,3.545,4800
8,Qwen3-1.7B,MIDDLE,3.684,3.277,3.766,2.900,3.407,4800
9,Llama3.2-1b,MIDDLE,3.139,2.327,3.315,2.150,2.733,4800


In [ ]:
end_df

,Model,conversation_stage,Human-Likeness,Continuity and Context Understanding,Tone and Clarity,Task Appropriateness,Overall_Mean,Tested Conversation Count
0,GPT-4.1,END,4.563,4.323,4.588,4.253,4.432,600
1,Virtuoso-Large,END,4.453,4.212,4.468,4.115,4.312,600
2,Llama3.1-8b,END,4.386,4.102,4.397,4.030,4.229,600
3,Llama-3.2-3B,END,4.357,4.068,4.373,4.002,4.200,600
4,Qwen3-4B,END,4.335,3.993,4.340,3.950,4.154,600
5,Phi-4-Mini,END,4.273,3.983,4.297,3.932,4.121,600
6,Qwen3-8B,END,4.182,3.967,4.212,3.885,4.062,600
7,Qwen3-1.7B,END,4.032,3.902,4.095,3.591,3.905,600
8,Gemini-2.5-Flash,END,4.085,3.577,4.177,3.433,3.818,600
9,Llama3.2-1b,END,2.910,2.033,3.173,1.930,2.512,600


In [ ]:
overall_df = (
    llm_judge_stagewise_df
    .sort_values("Overall_Mean", ascending=False)
    .reset_index(drop=True)
)

In [ ]:
Dataset.from_pandas(begin_df).push_to_hub(
    "Lakshan2003/customerservice-llm-judge-beginning-stage",
    private=False
)

Dataset.from_pandas(middle_df).push_to_hub(
    "Lakshan2003/customerservice-llm-judge-middle-stage",
    private=False
)

Dataset.from_pandas(end_df).push_to_hub(
    "Lakshan2003/customerservice-llm-judge-end-stage",
    private=False
)

Dataset.from_pandas(overall_df).push_to_hub(
    "Lakshan2003/customerservice-llm-judge-overall",
    private=False
)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 4.60kB / 4.60kB            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 5.66kB / 5.66kB            

                              : 100%|##########| 5.66kB / 5.66kB            

CommitInfo(commit_url='https://huggingface.co/datasets/Lakshan2003/customerservice-llm-judge-overall/commit/9823d927a9a9524d1f5d01fd44189d8a2b9457e0', commit_message='Upload dataset', commit_description='', oid='9823d927a9a9524d1f5d01fd44189d8a2b9457e0', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Lakshan2003/customerservice-llm-judge-overall', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Lakshan2003/customerservice-llm-judge-overall'), pr_revision=None, pr_num=None)

## Pairwise Evaluation Dataset Creation

In [ ]:
from datasets import load_dataset, Dataset
import pandas as pd

# configuration
N_PAIRS = 1000
RANDOM_SEED = 42
KEY_COLUMN = "conversation_id"

COLUMNS_TO_REMOVE = [
    "Human-Likeness",
    "Continuity and Context Understanding",
    "Tone and Clarity",
    "Task Appropriateness"
]

CONTEXT_COLUMNS = [
    "conversation_id",
    "conversation_stage",
    "instruction",
    "history",
    "history_summary",
    "client_question",
    "ground_truth"
]

# Baseline models (LLM + Human)
BASELINES = {
    "gpt4.1": {
        "repo": "Lakshan2003/GPT-4.1-customerservice-LLM-as-a-judge-data",
        "answer_col": "generated_answer_gpt_4_1"
    },
    "gemini-2.5-flash": {
        "repo": "Lakshan2003/gemini-2.5-flash-customerservice-LLM-as-a-judge-data",
        "answer_col": "generated_answer_gemini_2_5_flash"
    },
    "virtuoso-large": {
        "repo": "Lakshan2003/Virtuoso-large-customerservice-LLM-as-a-judge-data",
        "answer_col": "generated_answer_virtuoso_large"
    }
}

# SLM comparison models
SLM_MODELS = {
    "Qwen3-4B": {
        "repo": "Lakshan2003/Qwen3-4B-customerservice-LLM-as-a-judge-data",
        "col_name": "generated_answer_qwen3_4b"
    },
    "LLaMA-3.2-Instruct": {
        "repo": "Lakshan2003/Llama3.2-instruct-customerservice-LLM-as-a-judge-data",
        "col_name": "generated_answer_llama3_2"
    },
    "Phi-4-mini": {
        "repo": "Lakshan2003/Phi-4-mini-customerservice-LLM-as-a-judge-data",
        "col_name": "generated_answer_phi_4_mini"
    },
     "Qwen3-8B": {
        "repo": "Lakshan2003/Qwen3-8B-customerservice-LLM-as-a-judge-data",
        "col_name": "generated_answer_qwen3_8b"
    },
    "Llama3.1-8B": {
        "repo": "Lakshan2003/Llama3.1-8b-instruct-customerservice-LLM-as-a-judge-data",
        "col_name": "generated_answer_llama3_1_8b"
    },
    "Qwen3-1.7B": {
        "repo": "Lakshan2003/Qwen3-1.7B-customerservice-LLM-as-a-judge-data",
        "col_name": "generated_answer_qwen3_1_7b"
    },
    "Llama3.2-1B": {
        "repo": "Lakshan2003/Llama3.2-1B-instruct-customerservice-LLM-as-a-judge-data",
        "col_name": "generated_answer_llama3_2_1b"
    },

    "SmolLM3-3B": {
        "repo": "Lakshan2003/SmolLM3-3B-customerservice-evaldata",
        "col_name": "generated_answer_smollm3_3b"
    },
    "Gemma-3-4B": {
        "repo": "Lakshan2003/Gemma3-4B-instruct-customerservice-evaldata",
        "col_name": "generated_answer_gemma3_4b"
    },

}


# Load GPT-4.1 to select shared 1,000 IDs
print("Loading GPT-4.1 for shared sampling...")
ds_ref = load_dataset(BASELINES["gpt4.1"]["repo"], split="train")
df_ref = ds_ref.to_pandas()

df_ref = df_ref.drop(columns=COLUMNS_TO_REMOVE, errors="ignore")
df_ref = df_ref[CONTEXT_COLUMNS + ["generated_answer"]]

if len(df_ref) < N_PAIRS:
    raise ValueError("Not enough records to sample 1,000 shared instances.")

shared_ids = (
    df_ref[KEY_COLUMN]
    .sample(n=N_PAIRS, random_state=RANDOM_SEED)
    .tolist()
)

print(f"Selected {len(shared_ids)} shared conversation_ids")

# Build pairwise datasets
for baseline_name, baseline_cfg in BASELINES.items():

    print(f"\n=== Baseline: {baseline_name} ===")

    # Load baseline
    ds_base = load_dataset(baseline_cfg["repo"], split="train")
    df_base = ds_base.to_pandas()

    df_base = df_base.drop(columns=COLUMNS_TO_REMOVE, errors="ignore")
    df_base = df_base.rename(
        columns={"generated_answer": baseline_cfg["answer_col"]}
    )

    df_base = df_base[
        CONTEXT_COLUMNS + [baseline_cfg["answer_col"]]
    ]

    df_base = df_base[df_base[KEY_COLUMN].isin(shared_ids)].reset_index(drop=True)

    # Compare with each SLM
    for slm_name, slm_cfg in SLM_MODELS.items():

        print(f"Pairwise: {baseline_name} vs {slm_name}")

        ds_slm = load_dataset(slm_cfg["repo"], split="train")
        df_slm = ds_slm.to_pandas()

        df_slm = df_slm.drop(columns=COLUMNS_TO_REMOVE, errors="ignore")
        df_slm = df_slm.rename(
            columns={"generated_answer": slm_cfg["col_name"]}
        )

        df_slm = df_slm[
            [KEY_COLUMN, slm_cfg["col_name"]]
        ]

        df_slm = df_slm[df_slm[KEY_COLUMN].isin(shared_ids)]


        # Merge

        df_pairwise = pd.merge(
            df_base,
            df_slm,
            on=KEY_COLUMN,
            how="inner",
            validate="one_to_one"
        )

        if len(df_pairwise) != N_PAIRS:
            raise ValueError(
                f"Expected {N_PAIRS} rows, got {len(df_pairwise)}"
            )


        # Judge fields

        df_pairwise["judge_pass_1"] = None
        df_pairwise["judge_pass_2"] = None
        df_pairwise["final_result"] = None


        # Push to Hub

        out_repo = (
            f"Lakshan2003/pairwise-{baseline_name}-vs-{slm_name.lower()}"
        )

        hf_dataset = Dataset.from_pandas(
            df_pairwise,
            preserve_index=False
        )

        hf_dataset.push_to_hub(out_repo, private=False)

        print(f"Pushed → https://huggingface.co/datasets/{out_repo}")

print("\nAll baseline × SLM pairwise datasets created with the SAME 1,000 instances.")

Loading GPT-4.1 for shared sampling...
Selected 1000 shared conversation_ids

=== Baseline: gpt4.1 ===
Pairwise: gpt4.1 vs SmolLM3-3B


README.md:   0%|          | 0.00/541 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/42.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/36669 [00:00<?, ? examples/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 1.32MB / 1.32MB            

Pushed → https://huggingface.co/datasets/Lakshan2003/pairwise-virtuoso-large-vs-gemma-3-4b

All baseline × SLM pairwise datasets created with the SAME 1,000 instances.


In [ ]:
from datasets import load_dataset
import pandas as pd
import os

BASE_DIR = "/content/drive/MyDrive/fyp-2025/Datasets/"
os.makedirs(BASE_DIR, exist_ok=True)

PAIRWISE_DATASETS = [
    "Lakshan2003/pairwise-gpt4.1-vs-qwen3-4b",
    "Lakshan2003/pairwise-gpt4.1-vs-llama-3.2-instruct",
    "Lakshan2003/pairwise-gpt4.1-vs-phi-4-mini",

    "Lakshan2003/pairwise-gemini-2.5-flash-vs-qwen3-4b",
    "Lakshan2003/pairwise-gemini-2.5-flash-vs-llama-3.2-instruct",
    "Lakshan2003/pairwise-gemini-2.5-flash-vs-phi-4-mini",

    "Lakshan2003/pairwise-virtuoso-large-vs-qwen3-4b",
    "Lakshan2003/pairwise-virtuoso-large-vs-llama-3.2-instruct",
    "Lakshan2003/pairwise-virtuoso-large-vs-phi-4-mini",

    # GPT-4.1 vs new models
    "Lakshan2003/pairwise-gpt4.1-vs-qwen3-8b",
    "Lakshan2003/pairwise-gpt4.1-vs-llama3.1-8b",
    "Lakshan2003/pairwise-gpt4.1-vs-qwen3-1.7b",
    "Lakshan2003/pairwise-gpt4.1-vs-llama3.2-1b",

    # Gemini-2.5-Flash vs new models
    "Lakshan2003/pairwise-gemini-2.5-flash-vs-qwen3-8b",
    "Lakshan2003/pairwise-gemini-2.5-flash-vs-llama3.1-8b",
    "Lakshan2003/pairwise-gemini-2.5-flash-vs-qwen3-1.7b",
    "Lakshan2003/pairwise-gemini-2.5-flash-vs-llama3.2-1b",

    # Virtuoso-Large vs new models
    "Lakshan2003/pairwise-virtuoso-large-vs-qwen3-8b",
    "Lakshan2003/pairwise-virtuoso-large-vs-llama3.1-8b",
    "Lakshan2003/pairwise-virtuoso-large-vs-qwen3-1.7b",
    "Lakshan2003/pairwise-virtuoso-large-vs-llama3.2-1b",

    # Virtuoso-Large vs new SLMs
    "Lakshan2003/pairwise-virtuoso-large-vs-gemma-3-4b",
    "Lakshan2003/pairwise-virtuoso-large-vs-smollm3-3b",

    # Gemini-2.5-Flash vs new SLMs
    "Lakshan2003/pairwise-gemini-2.5-flash-vs-gemma-3-4b",
    "Lakshan2003/pairwise-gemini-2.5-flash-vs-smollm3-3b",

    # GPT-4.1 vs new SLMs
    "Lakshan2003/pairwise-gpt4.1-vs-gemma-3-4b",
    "Lakshan2003/pairwise-gpt4.1-vs-smollm3-3b",
]


for repo_id in PAIRWISE_DATASETS:
    print(f"Saving {repo_id}")

    ds = load_dataset(repo_id, split="train")
    df = ds.to_pandas()

    file_name = repo_id.split("/")[-1] + ".csv"
    out_path = os.path.join(BASE_DIR, file_name)

    df.to_csv(out_path, index=False)

    print(f"Saved to {out_path}")

print("All pairwise datasets saved into the same Drive folder successfully.")

Saving Lakshan2003/pairwise-virtuoso-large-vs-gemma-3-4b


README.md:   0%|          | 0.00/775 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.32M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Saved to /content/drive/MyDrive/fyp-2025/Datasets/pairwise-virtuoso-large-vs-gemma-3-4b.csv
Saving Lakshan2003/pairwise-virtuoso-large-vs-smollm3-3b


...

Saved to /content/drive/MyDrive/fyp-2025/Datasets/pairwise-gemini-2.5-flash-vs-smollm3-3b.csv
Saving Lakshan2003/pairwise-gpt4.1-vs-gemma-3-4b


README.md:   0%|          | 0.00/768 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.32M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Saved to /content/drive/MyDrive/fyp-2025/Datasets/pairwise-gpt4.1-vs-gemma-3-4b.csv
Saving Lakshan2003/pairwise-gpt4.1-vs-smollm3-3b
Saved to /content/drive/MyDrive/fyp-2025/Datasets/pairwise-gpt4.1-vs-smollm3-3b.csv
All pairwise datasets saved into the same Drive folder successfully.


## Semantic and Lexical Similarity results

In [ ]:
import pandas as pd
import glob
import os

BASE_DIR = "/content/drive/MyDrive/fyp-2025/Datasets/Semantic_Lexical_Similarity_Results"
N_TESTED = 36669

MODEL_NAME_MAP = {
    "metrics_virtuso_large.csv":         "Virtuoso-Large",
    "metrics_SmolLM3-3B-instruct.csv":   "SmolLM-3B-Instruct",
    "metrics_Qwen3-4B-instruct.csv":     "Qwen-3-4B-Instruct",
    "metrics_Phi-4-mini.csv":            "Phi-4-Mini",
    "metrics_Llama-3.2_3B-instruct.csv": "LLaMA-3.2-3B-Instruct",
    "metrics_gpt-4.1.csv":               "GPT-4.1",
    "metrics_Gemma3-4B.csv":             "Gemma-3-4B",
    "metrics_gemini-2.5-flash.csv":      "Gemini-2.5-Flash",


    "metrics_Qwen3-8B-instruct.csv":     "Qwen-3-8B-Instruct",
    "metrics_Llama-3.1_8B-instruct.csv":  "LLaMA-3.1-8B-Instruct",
    "metrics_Qwen3-1.7B-instruct.csv":   "Qwen-3-1.7B-Instruct",
    "metrics_Llama-3.2_1B-instruct.csv": "LLaMA-3.2-1B-Instruct",
}

METRIC_RENAME_MAP = {
    "Cosine Similarity (mean)": "Cosine Similarity (all-mpnet-base-v2)",
    "Cosine Similarity sentenceEmbeddings (mean)": "Cosine Similarity (all-mpnet-base-v2)",
    "Sentence Embedding Cosine Similarity (mean)": "Cosine Similarity (all-mpnet-base-v2)",
    "Cosine Similarity (Sentence Embedding- all-mpnet-base-v2)": "Cosine Similarity (all-mpnet-base-v2)",
}

QA_METRICS = [
    "ROUGE-1",
    "ROUGE-2",
    "ROUGE-L",
    "METEOR",
    "Cosine Similarity (all-mpnet-base-v2)",
    "BERTScore F1 (mean)",
    "BARTScore (mean)",
]

dfs = []

for file in glob.glob(os.path.join(BASE_DIR, "metrics_*.csv")):
    fname = os.path.basename(file)
    if fname not in MODEL_NAME_MAP:
        continue

    df = pd.read_csv(file)

    df["metric"] = df["metric"].astype(str).str.strip()
    df["metric"] = df["metric"].replace(METRIC_RENAME_MAP)

    df = df[df["metric"].isin(QA_METRICS)]
    df["model"] = MODEL_NAME_MAP[fname]

    dfs.append(df)

all_metrics = pd.concat(dfs, ignore_index=True)

qa_matrix = (
    all_metrics
    .pivot(index="model", columns="metric", values="value")
    .reset_index()
    .sort_values("model")
)

metric_cols = qa_matrix.columns.drop("model")
qa_matrix[metric_cols] = qa_matrix[metric_cols].applymap(lambda x: f"{float(x):.4f}")

qa_matrix["num_tested_samples"] = N_TESTED

save_path = os.path.join(BASE_DIR, "qa_metrics_matrix_clean_models.csv")
qa_matrix.to_csv(save_path, index=False)

/tmp/ipython-input-2761155826.py:69: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  qa_matrix[metric_cols] = qa_matrix[metric_cols].applymap(lambda x: f"{float(x):.4f}")


In [ ]:
from huggingface_hub import HfApi, upload_file

REPO_ID = "Lakshan2003/customerservice_qa_semantic_lexical_results"
FILE_PATH = "/content/drive/MyDrive/fyp-2025/Datasets/Semantic_Lexical_Similarity_Results/qa_metrics_matrix_clean_models.csv"

api = HfApi()

api.create_repo(
    repo_id=REPO_ID,
    repo_type="dataset",
    exist_ok=True
)

upload_file(
    path_or_fileobj=FILE_PATH,
    path_in_repo="qa_metrics_matrix.csv",
    repo_id=REPO_ID,
    repo_type="dataset"
)

print(f"Pushed to https://huggingface.co/datasets/{REPO_ID}")


Pushed to https://huggingface.co/datasets/Lakshan2003/customerservice_qa_semantic_lexical_results


In [ ]:
qa_matrix

metric,model,BARTScore (mean),BERTScore F1 (mean),Cosine Similarity (all-mpnet-base-v2),METEOR,ROUGE-1,ROUGE-2,ROUGE-L,num_tested_samples
0,GPT-4.1,-2.5145,0.8994,0.6749,0.3685,0.4029,0.1678,0.3038,36669
1,Gemini-2.5-Flash,-2.6409,0.8942,0.6234,0.3110,0.3675,0.1422,0.2771,36669
2,Gemma-3-4B,-3.0766,0.8752,0.5134,0.2782,0.2883,0.0764,0.2024,36669
3,LLaMA-3.1-8B-Instruct,-2.2332,0.9134,0.7051,0.4569,0.4794,0.2627,0.3940,36669
4,LLaMA-3.2-1B-Instruct,-2.7060,0.8821,0.5909,0.3032,0.3298,0.1053,0.2332,36669
5,LLaMA-3.2-3B-Instruct,-2.2655,0.9121,0.6958,0.4471,0.4697,0.2514,0.3842,36669
6,Phi-4-Mini,-2.2872,0.9107,0.6891,0.4303,0.4607,0.2420,0.3747,36669
7,Qwen-3-1.7B-Instruct,-2.3096,0.9096,0.6731,0.4138,0.4508,0.2347,0.3697,36669
8,Qwen-3-4B-Instruct,-2.2311,0.9137,0.6972,0.4455,0.4768,0.2636,0.3959,36669
9,Qwen-3-8B-Instruct,-2.4970,0.8995,0.6621,0.3792,0.4041,0.1795,0.3121,36669
